In [28]:
import pandas as pd 
import seaborn as sns
import os 

In [29]:
import os
import cv2
import numpy as np
import torch
import torch.nn as nn
from PIL import Image
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from torchvision.models import resnet18, ResNet18_Weights
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# --- دالة تسويد الحواف لمنع الغش نهائياً ---
def blackout_edges(pil_img):
    # تحويل الصورة لـ NumPy
    img = np.array(pil_img)
    
    # تحديد عدد البكسلات اللي هنطيرها من الأطراف (مثلاً 35 بكسل من كل جنب)
    # ده هيضمن إن أي حرف R أو L أو كتابة اتحولت لأسود تماماً
    pad = 35 
    
    # تسويد الحواف
    img[:pad, :, :] = 0      # فوق
    img[-pad:, :, :] = 0     # تحت
    img[:, :pad, :] = 0      # شمال
    img[:, -pad:, :] = 0     # يمين
    
    return Image.fromarray(img)

# 1. تحديد الـ Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 2. تصليح المسارات القاتلة (تم تعديلها لتكون صحيحة ومباشرة)
DATA_DIR = r"D:\__Projects\Graduation---Project\DL\Chest-x-ray\data" 
TRAIN_DIR = os.path.join(DATA_DIR, 'train')
TEST_DIR = os.path.join(DATA_DIR, 'test')

# 3. الـ Transforms الجديدة مع الـ Blackout
train_transforms = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.Lambda(blackout_edges),     # تسويد الحروف تماماً وهي في حجمها الكبير
    transforms.CenterCrop((224, 224)),     # الـ Crop العادي بتاعنا
    transforms.RandomAutocontrast(p=0.5),  
    transforms.ColorJitter(brightness=0.2, contrast=0.2), 
    transforms.RandomRotation(degrees=15), 
    transforms.RandomHorizontalFlip(), 
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_transforms = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.Lambda(blackout_edges),     # لازم في الـ test والـ API تعمل نفس الكلام
    transforms.CenterCrop((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# تحميل الداتا والـ Loaders
train_dataset = datasets.ImageFolder(root=TRAIN_DIR, transform=train_transforms)
test_dataset = datasets.ImageFolder(root=TEST_DIR, transform=test_transforms)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0) # خليه 0 لو الـ workers بيعملوا crash
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=0)

In [30]:
from torchvision.models import resnet18, ResNet18_Weights
import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
weights= ResNet18_Weights.IMAGENET1K_V1
model= resnet18(weights=weights)

device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
in_features= model.fc.in_features
model.fc= nn.Sequential(
    nn.BatchNorm1d(in_features),
    nn.Dropout(p=0.5),
    nn.Linear(in_features, 1)
)
model = model.to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
# تذكر تحديد الـ device لو مش متعرّف فوق
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

epoch = 5
for e in range(epoch):
    # ==================== phase 1: TRAINING ====================
    model.train()
    running_loss = 0.0
    
    for imgs, labels in train_loader:
        imgs = imgs.to(device)
        # تحويل الـ labels لـ Float لأن BCEWithLogitsLoss بتطلب كده
        labels = labels.to(device).float().unsqueeze(1) 
        
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * imgs.size(0)
        
    epoch_loss = running_loss / len(train_loader.dataset)
    print(f"\nEpoch [{e+1}/{epoch}] | Train Loss: {epoch_loss:.4f}")
    
    model.eval()
    test_loss = 0.0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for imgs, labels in test_loader:
            imgs = imgs.to(device)
            labels = labels.to(device).float().unsqueeze(1)
            
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            test_loss += loss.item() * imgs.size(0)
            probs = torch.sigmoid(outputs)
            preds = (probs > 0.5).int()
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            
    epoch_test_loss = test_loss / len(test_loader.dataset)
    
    acc = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds, zero_division=0)
    recall = recall_score(all_labels, all_preds, zero_division=0)
    f1 = f1_score(all_labels, all_preds, zero_division=0)
    
    print(f"Test Loss: {epoch_test_loss:.4f} | Accuracy: {acc:.4f} | Precision: {precision:.4f} | Recall: {recall:.4f} | F1-Score: {f1:.4f}")


Epoch [1/5] | Train Loss: 0.2133
Test Loss: 0.2044 | Accuracy: 0.9119 | Precision: 0.8923 | Recall: 0.9769 | F1-Score: 0.9327

Epoch [2/5] | Train Loss: 0.1105
Test Loss: 0.2642 | Accuracy: 0.9103 | Precision: 0.8761 | Recall: 0.9974 | F1-Score: 0.9329

Epoch [3/5] | Train Loss: 0.1058
Test Loss: 0.1910 | Accuracy: 0.9391 | Precision: 0.9131 | Recall: 0.9974 | F1-Score: 0.9534

Epoch [4/5] | Train Loss: 0.0885
Test Loss: 0.3029 | Accuracy: 0.8814 | Precision: 0.8420 | Recall: 0.9974 | F1-Score: 0.9131

Epoch [5/5] | Train Loss: 0.0909
Test Loss: 0.3079 | Accuracy: 0.8782 | Precision: 0.8384 | Recall: 0.9974 | F1-Score: 0.9110


In [31]:
checkpoint = {
    "model_state_dict": model.state_dict(),
    "model_name": "resnet18",
    "num_classes": 1,
    "in_features": model.fc[2].in_features,
    "dropout": 0.5,
    "class_mapping": {"Normal": 0, "Pneumonia": 1},
    "threshold": 0.5,
    "image_size": 224,
    "normalization_mean": [0.485, 0.456, 0.406],
    "normalization_std": [0.229, 0.224, 0.225]}

torch.save(checkpoint, r"D:\__Projects\Graduation---Project\app\models\chest_xray_best_model.pth")

In [33]:
import cv2
import numpy as np
import torch
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from torchvision import transforms
from PIL import Image
from torchvision.models import resnet18

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

checkpoint_path = r"D:\__Projects\Graduation---Project\app\models\chest_xray_best_model.pth"
checkpoint = torch.load(checkpoint_path, map_location=device)

model = resnet18(weights=None)  
in_features = model.fc.in_features

model.fc = torch.nn.Sequential(
    torch.nn.BatchNorm1d(in_features),
    torch.nn.Dropout(p=checkpoint["dropout"]),
    torch.nn.Linear(in_features, checkpoint["num_classes"])
)

# تحميل الأوزان وتجهيز الموديل للاختبار
model.load_state_dict(checkpoint["model_state_dict"])
model = model.to(device)
model.eval()

# 3. تجهيز التحويلات (Transforms) للصورة المدخلة
# نستخدم نفس التحويلات الخاصة بالـ validation المعرفة عندك
cam_transform = transforms.Compose([
    transforms.Resize((checkpoint["image_size"], checkpoint["image_size"])),
    transforms.ToTensor(),
    transforms.Normalize(checkpoint["normalization_mean"], checkpoint["normalization_std"])
])

image_path = r"D:\__Projects\Graduation---Project\DL\Chest-x-ray\data\test\PNEUMONIA\BACTERIA-518323-0002.jpeg" 
rgb_img = Image.open(image_path).convert('RGB')

# تجهيز الصورة كـ Tensor للموديل [1, 3, 224, 224]
input_tensor = cam_transform(rgb_img).unsqueeze(0).to(device)

input_image_resized = rgb_img.resize((checkpoint["image_size"], checkpoint["image_size"]))
input_image_np = np.float32(input_image_resized) / 255.0

target_layers = [model.layer4[-1]]

# إنشاء كائن الـ Grad-CAM
cam = GradCAM(model=model, target_layers=target_layers)

# بما أن الموديل تصنيف ثنائي ومخرجه واحد (Binary Classification مع Logits)
# فإن القيمة المرتفعة تعني "Pneumonia" والقيمة المنخفضة تعني "Normal"
# هنا نخبر الـ CAM بالتركيز على النتيجة التي يخرجها الموديل مباشرة
targets = [ClassifierOutputTarget(0)] 

# 6. توليد الخريطة الحرارية (Heatmap)
grayscale_cam = cam(input_tensor=input_tensor, targets=targets)
grayscale_cam = grayscale_cam[0, :] # استخراج الصورة الأولى من الباتش

# 7. دمج الخريطة الحرارية مع الصورة الأصلية
visualization = show_cam_on_image(input_image_np, grayscale_cam, use_rgb=True)

# 8. عرض النتيجة وحفظها
# تحويل النتيجة إلى صورة PIL لعرضها وحفظها بسهولة
output_image = Image.fromarray(visualization)
output_image.show() # ستفتح لك الصورة فوراً

# حفظ الصورة في المجلد الخاص بالمشروع إذا أردت
output_image.save(r"D:\__Projects\Graduation---Project\app\models\grad_cam_result.png")
print("تم توليد وحفظ صورة Grad-CAM بنجاح!")

تم توليد وحفظ صورة Grad-CAM بنجاح!
